# Multilingual Keyword Spotting — Embedding-First (v2)

**Course**: Intro to Deep Learning &nbsp;|&nbsp; **Due**: May 14, 2026 (with extension)

## What this notebook is (and what changed from v1)

This is a **full pivot** from the previous notebook. v1 trained a DS-CNN end-to-end on a mostly-TTS dataset and stalled near the accuracy ceiling identified by Lin et al. (ICASSP 2020) for "full models trained on synthetic data." v2 follows the architecture of **Mazumder et al., Interspeech 2021** (*Few-Shot Keyword Spotting in Any Language*), scaled down to a TinyML backbone.

### Pipeline

```
MSWC real audio (9 langs × top-50 keywords each)
        │
        ▼
DS-CNN classifier  (~400-500K params, 256-dim penultimate)
        │
        ▼  freeze
multilingual embedding  ──▶  Section 7: t-SNE by language
        │
        ▼
5-shot head per target keyword (Section 8: ~180 head models across 9 langs)
        │
        ▼
F1 distribution per language  (Mazumder Fig. 2 reproduction at <500K params)
```

### Contribution

Mazumder et al. (2021) achieved 5-shot F1 ≈ 0.75 with an **11M-parameter** EfficientNet-B0 embedding. They explicitly listed *"a smaller embedding representation via knowledge distillation for on-device deployment"* as future work. This notebook does that work: same training recipe, same evaluation protocol, but a DS-CNN backbone roughly **20× smaller**, suitable for ESP32-class hardware.

### What this notebook does *not* do
- No quantization / ONNX export — those live in `multilingual_kws_export.ipynb` to keep this notebook free of the TensorFlow / `onnx2tf` / `qnnpack` dependency chain that broke v1.
- No TTS in the training loop. TTS reappears only in optional Section 8b as a deployment-time *enrollment* tool (Lin et al.'s validated use case).
- No language-conditioned keyword head, no misrouting experiment — v1 ideas demoted or removed in favor of the simpler embedding approach.

### Key references
- Mazumder et al., *Interspeech 2021* — multilingual embedding + few-shot heads
- Mazumder et al., *NeurIPS 2021* — MSWC dataset
- Lin et al., *ICASSP 2020* — limits of synthetic speech for KWS
- Zhang et al., 2017 — DS-CNN ("Hello Edge") backbone
- Lei et al., *TASLP 2023* — embedding-dim sweep informed our 256-dim choice


In [ ]:
# ── Install pinned dependencies (Colab) ───────────────────────────────────────
# NO tensorflow, NO onnx2tf, NO qnnpack. Quantization/export live in the
# separate `multilingual_kws_export.ipynb` notebook.
# huggingface_hub: direct HF shard access (load_dataset is blocked for this
# dataset due to HF trust_remote_code policy changes).
%pip install -q "torchaudio>=2.0" "scikit-learn>=1.3" "seaborn>=0.13" "huggingface_hub>=0.20"
# After the first run on a fresh Colab: Runtime > Restart, then skip this cell.

In [9]:
import os
try:
    import google.colab  # noqa: F401
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab and not os.path.exists('/content/polyglot-kws'):
        !git clone https://github.com/verilog-indeed/polyglot-kws /content/polyglot-kws

In [10]:
import os, io, json, random, warnings, math, time, tempfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, roc_curve, auc
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# No HuggingFace I/O happens in this notebook — fetch_kws_data.py does
# all dataset downloading and feature extraction.  We only load .npy files
# from CACHE_DIR (Drive in Colab, local dir otherwise).

print('Imports OK. Device:', device)


Imports OK. Device: cpu


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DEBUG = True   # flip to False for the real training run

LANGUAGES = ['en', 'de', 'fr', 'ca', 'fa', 'es', 'it', 'nl', 'rw']
LANG_NAMES = {
    'en': 'English', 'de': 'German',  'fr': 'French',  'ca': 'Catalan',
    'fa': 'Persian', 'es': 'Spanish', 'it': 'Italian', 'nl': 'Dutch',
    'rw': 'Kinyarwanda',
}

# Audio / spectrogram — 49×40 log-mel, matching TF Lite Micro microfrontend.
# center=False + n_fft=win_length=640 gives exactly 49 frames:
#   1 + floor((16000 - 640) / 320) = 49
SAMPLE_RATE = 16_000
N_MELS      = 40
N_FFT       = 640
WIN_LENGTH  = 640
HOP_LENGTH  = 320
F_MIN       = 20
F_MAX       = 8_000

# Embedding training
TOP_K_PER_LANG       = 50
SAMPLES_PER_CLASS    = 400   # cap per (lang, word) for both training and heldout
NOISE_FRACTION       = 0.10
EMBEDDING_DIM        = 256
N_HELDOUT_KEYWORDS   = 20

# Training
BATCH_SIZE     = 128
EPOCHS         = 30
LR             = 3e-3
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTH   = 0.1

# 5-shot head
N_SHOT             = 5
N_UNKNOWN_PER_HEAD = 128
N_NOISE_PER_HEAD   = 25
HEAD_EPOCHS        = 40
HEAD_LR            = 1e-2
F1_THRESHOLD       = 0.8

# ── Debug overrides ────────────────────────────────────────────────────────────
if DEBUG:
    LANGUAGES            = ['en', 'de']
    TOP_K_PER_LANG       = 5
    SAMPLES_PER_CLASS    = 40
    N_HELDOUT_KEYWORDS   = 2
    BATCH_SIZE           = 32
    EPOCHS               = 3
    HEAD_EPOCHS          = 5
    N_UNKNOWN_PER_HEAD   = 20
    N_NOISE_PER_HEAD     = 5
    print('*** DEBUG MODE — small data, fast run, results not meaningful ***')

# ── Colab / local path setup ──────────────────────────────────────────────────
_cache_name = 'kws_cache_debug' if DEBUG else 'kws_cache'
_cache_name = 'kws_cache_debug' if DEBUG else 'kws_cache'

if _in_colab:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT_DIR = '/content/drive/MyDrive/'
    print('Colab detected — cache routed to Google Drive.')
    ROOT_DIR = '/content/drive/MyDrive/'
    print('Colab detected — cache routed to Google Drive.')
    REPO_DIR = '/content/polyglot-kws/'
else:
    ROOT_DIR = './'
    print('Local run.')
    ROOT_DIR = './'
    print('Local run.')
    REPO_DIR = './'

CACHE_DIR = Path(f'{ROOT_DIR}/{_cache_name}')
print('Cache at:', CACHE_DIR.resolve())
CACHE_DIR = Path(f'{ROOT_DIR}/{_cache_name}')
print('Cache at:', CACHE_DIR.resolve())
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FEATS_DIR   = CACHE_DIR / 'feats'
INVENT_PATH = CACHE_DIR / 'keyword_inventory.json'
CKPT_PATH   = CACHE_DIR / 'embedding_best.pt'
FEATS_DIR.mkdir(exist_ok=True)

print(f'Languages : {LANGUAGES}')
print(f'Classes   : up to {len(LANGUAGES) * TOP_K_PER_LANG} + 1 (noise)')
print(f'Cache dir : {CACHE_DIR}')

*** DEBUG MODE — small data, fast run, results not meaningful ***
Local run.
Cache at: D:\studies\phd\intro_deep_learning\final_project\kws_cache_debug
Languages : ['en', 'de']
Classes   : up to 10 + 1 (noise)
Cache dir : kws_cache_debug


> **Before running this notebook**, run `fetch_kws_data.py` once. The repo is already cloned by the cell above, so just run:
>
> ```bash
> # Upload mswc-metadata.json to the root of your Drive first, then:
> !python /content/polyglot-kws/fetch_kws_data.py \
>     --metadata /content/drive/MyDrive/mswc-metadata.json \
>     --root     /content/drive/MyDrive/kws_cache
> ```
>
The script downloads MSWC shards from HuggingFace, decodes audio, computes 49×40 log-mel features, and saves `feats/{kind}/{lang}/{word}.npy` + `keyword_inventory.json` to Drive. **The notebook never contacts HuggingFace** — it loads everything from Drive cache.
>
> For a quick smoke-test before the full run, add `--debug` (2 langs, top-5 words, 40 samples).

In [ ]:
!python {REPO_DIR}/fetch_kws_data.py \
--metadata {ROOT_DIR}/mswc-metadata.json \
--root {CACHE_DIR} \
--langs {' '.join(LANGUAGES)} \
--top-k {TOP_K_PER_LANG} \
--n-heldout {N_HELDOUT_KEYWORDS} \
--samples {SAMPLES_PER_CLASS} \
--split train

---
## Section 2 — MSWC keyword inventory

For each of the 9 languages we pick the top **50 most frequent** words (≥3 chars) from `mswc-metadata.json` wordcounts as our embedding training classes. We also reserve a **held-out keyword bank** of 20 words per language that the embedding never sees during training — these are the targets of the Section 8 5-shot evaluation.

The inventory is saved to `keyword_inventory.json` so subsequent runs are deterministic. The `build_inventory` function below reads this file directly from Drive; the HuggingFace streaming path is never triggered once the script has run.

**Why inventory-first**: MSWC is uneven across languages (English ~1957 hours, Persian ~7.6 hours). If low-resource languages can't supply enough heldout candidates, we want to know before training starts.

In [ ]:
# ── Load the keyword inventory built by fetch_kws_data.py ────────────────────
# The notebook never touches HuggingFace.  fetch_kws_data.py reads
# mswc-metadata.json, ranks each language's words by their true clip count
# (len(filenames[word])), and saves keyword_inventory.json + per-(lang, word)
# .npy feature files to CACHE_DIR.  Here we just load and sanity-check.

if not INVENT_PATH.exists():
    raise FileNotFoundError(
        f'No inventory at {INVENT_PATH}.'
        f'Run fetch_kws_data.py once before this notebook, e.g.:'
        f'  !python /content/polyglot-kws/fetch_kws_data.py \\'
        f'      --metadata /content/drive/MyDrive/mswc-metadata.json \\'
        f'No inventory at {INVENT_PATH}.'
        f'Run fetch_kws_data.py once before this notebook, e.g.:'
        f'  !python /content/polyglot-kws/fetch_kws_data.py \\'
        f'      --metadata /content/drive/MyDrive/mswc-metadata.json \\'
        f'      --root     {CACHE_DIR}'
        + ('  --debug' if DEBUG else '')
    )

inventory = json.loads(INVENT_PATH.read_text(encoding='utf-8'))

missing = [l for l in LANGUAGES if l not in inventory]
if missing:
    raise RuntimeError(
        f'Inventory is missing languages {missing}. '
        f'Re-run fetch_kws_data.py with --langs {" ".join(LANGUAGES)}.'
    )

print(f'Loaded inventory from {INVENT_PATH}')
print(f'{"lang":6s} {"#train":>7s} {"#heldout":>9s} {"min clips":>10s} {"max clips":>10s}')
print(f'{"lang":6s} {"#train":>7s} {"#heldout":>9s} {"min clips":>10s} {"max clips":>10s}')
for lang in LANGUAGES:
    inv     = inventory[lang]
    train   = inv.get('training', [])
    heldout = inv.get('heldout', [])
    counts  = inv.get('counts', {})
    tc = [counts.get(w, 0) for w in train]
    print(f'{lang:6s} {len(train):7d} {len(heldout):9d} '
          f'{(min(tc) if tc else 0):10d} {(max(tc) if tc else 0):10d}')


---
## Section 3 — Log-mel feature extraction

We convert each 1-second clip to a **49 × 40 log-mel spectrogram** (matching TF Lite Micro's microfrontend convention used by Mazumder et al., so our results are directly comparable to theirs).

**Pipeline**: waveform → STFT (40 ms window, 20 ms hop) → power → 40-band Mel filterbank → log → normalize to ≈ [0, 1].

**Why log-mel and not raw waveform / MFCCs**:
- **Raw waveform** has 16,000 numbers/sec with complex long-range dependencies. Hard for small CNNs to learn from.
- **MFCCs** apply a DCT on top of log-mel and discard high-order coefficients. The DCT decorrelates features — useful for GMM-HMM models, harmful for CNNs that can learn their own decorrelation given the full input. Modern KWS papers universally use log-mel.

**SpecAugment** masks small bands of time and frequency during training as a regularizer (Park et al. 2019). Applied only in train mode.

In [ ]:
# Mel transforms live on the same device as the training data so feature
# extraction during streaming runs on GPU when available.
#
# center=False is required to get exactly 49 frames:
#   frames = 1 + floor((16000 - n_fft) / hop_length)
#          = 1 + floor((16000 - 640)  / 320) = 49
# center=True (default) pads n_fft//2 on each side → 51 frames.
_mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH, n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX,
    power=2.0, center=False,
).to(device)
_amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80).to(device)

# SpecAugment for training-time regularization
spec_augment = nn.Sequential(
    T.FrequencyMasking(freq_mask_param=8),
    T.TimeMasking(time_mask_param=10),
)


def waveform_to_logmel(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    """
    Args:
        waveform : 1-D or 2-D tensor (mono assumed if 2-D, single channel)
        sr       : source sample rate
    Returns:
        spec     : (1, 49, 40) float32 tensor in approximately [0, 1]
    """
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = T.Resample(sr, SAMPLE_RATE)(waveform)

    target_len = SAMPLE_RATE
    if waveform.shape[-1] < target_len:
        waveform = F.pad(waveform, (0, target_len - waveform.shape[-1]))
    else:
        waveform = waveform[..., :target_len]

    waveform = waveform.to(device)
    mel = _mel_transform(waveform)        # (1, 40, 49)
    mel = _amplitude_to_db(mel)
    mel = mel.squeeze(0).T                # (49, 40)
    mel = (mel + 80.0) / 80.0
    return mel.unsqueeze(0).cpu()         # (1, 49, 40)


# ── Sanity check + visualization ───────────────────────────────────────────────
test_wav = torch.randn(SAMPLE_RATE) * 0.05
spec     = waveform_to_logmel(test_wav, SAMPLE_RATE)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(spec.squeeze().numpy().T, aspect='auto', origin='lower', cmap='magma')
axes[0].set_xlabel('Time frame (×20 ms)'); axes[0].set_ylabel('Mel bin')
axes[0].set_title(f'Log-mel  shape={tuple(spec.shape)}')

aug = spec_augment(spec).squeeze().numpy().T
axes[1].imshow(aug, aspect='auto', origin='lower', cmap='magma')
axes[1].set_xlabel('Time frame (×20 ms)'); axes[1].set_ylabel('Mel bin')
axes[1].set_title('After SpecAugment (training only)')
plt.tight_layout(); plt.show()
assert spec.shape == (1, 49, 40), spec.shape
print('Feature extraction OK.')

---
## Section 4 — Embedding dataset

`fetch_kws_data.py` has already decoded every MSWC clip and saved 49×40 log-mel features to `feats/<kind>/<lang>/<word>.npy` on Drive. `stream_and_cache` below detects the pre-built files and skips HuggingFace entirely.

The cached arrays are float16 to halve disk I/O.

We also load a **held-out keyword bank** for Section 8, plus generate a **background-noise pool** synthetically.

**Balanced sampling**: MSWC is uneven across languages — English keywords may have 400 clips while Kinyarwanda keywords have 40. A naive shuffle would give English 10× the gradient signal. We use `WeightedRandomSampler` (Mazumder et al. recipe) to assign each sample weight `1 / class_size`, so every class contributes equally to every epoch regardless of clip count.

**Train / val split**: random 90/10 *within* each (lang, word) class. Balancing is applied only to the training split; the val split uses sequential order so per-language accuracy is reported on the true class distribution.

In [ ]:
# ── Load pre-computed features from Drive cache ─────────────────────────────
# fetch_kws_data.py has decoded every (lang, word) into a float16 .npy at
#   FEATS_DIR/{kind}/{lang}/{word}.npy   shape (N, 1, 49, 40)
# where N <= min(SAMPLES_PER_CLASS, clips available in metadata).
# We load each into memory; missing files (rare low-resource words) are
# reported but do not abort.

def _cache_path(lang: str, word: str, kind: str) -> Path:
    safe = word.replace('/', '_')
    return FEATS_DIR / kind / lang / f'{safe}.npy'


def load_cached(lang: str, words, kind: str) -> dict:
    """Load all available {word: ndarray} for a (lang, kind) split.
    Skips and warns about words with no cached .npy."""
    out, missing = {}, []
    for w in words:
        p = _cache_path(lang, w, kind)
        if p.exists():
            out[w] = np.load(p)
        else:
            missing.append(w)
    if missing:
        print(f'  [{lang}/{kind}] {len(missing)} words missing from cache: '
              f'{missing[:5]}{"..." if len(missing) > 5 else ""}')
    return out


train_cache, heldout_cache = {}, {}
for lang in LANGUAGES:
    inv = inventory[lang]
    train_cache[lang]   = load_cached(lang, inv['training'], 'train')
    heldout_cache[lang] = load_cached(lang, inv['heldout'],  'heldout')
    n_train = sum(arr.shape[0] for arr in train_cache[lang].values())
    n_held  = sum(arr.shape[0] for arr in heldout_cache[lang].values())
    print(f'  [{lang}] {len(train_cache[lang])}/{len(inv["training"])} train words '
          f'({n_train} samples), '
          f'{len(heldout_cache[lang])}/{len(inv["heldout"])} heldout words '
          f'({n_held} samples)')

total_train = sum(arr.shape[0] for d in train_cache.values()  for arr in d.values())
total_held  = sum(arr.shape[0] for d in heldout_cache.values() for arr in d.values())
print(f'Total training samples loaded: {total_train:,}')
print(f'Total training samples loaded: {total_train:,}')
print(f'Total held-out samples loaded: {total_held:,}')

if total_train == 0:
    raise RuntimeError(
        f'No cached features found under {FEATS_DIR}. '
        f'Run fetch_kws_data.py first (see Section 2).'
    )


In [ ]:
# ── Synthetic background-noise pool ───────────────────────────────────────────
# 10% of training samples are "noise" — this gives the embedding a
# language-agnostic "no keyword present" class. We blend three noise types:
#   (a) pure silence (zeros)
#   (b) low-amplitude white noise
#   (c) low-amplitude pink-ish noise (1/f via cumulative sum)
# Cheap, deterministic, no extra dependencies.

def make_noise_pool(n: int) -> np.ndarray:
    rng = np.random.RandomState(SEED)
    out = []
    for _ in range(n):
        kind = rng.choice(['silence', 'white', 'pink'])
        if kind == 'silence':
            w = np.zeros(SAMPLE_RATE, dtype=np.float32)
        elif kind == 'white':
            w = rng.randn(SAMPLE_RATE).astype(np.float32) * 0.005
        else:
            w = np.cumsum(rng.randn(SAMPLE_RATE)).astype(np.float32)
            w = (w - w.mean()) / (np.std(w) + 1e-9) * 0.005
        spec = waveform_to_logmel(torch.from_numpy(w), SAMPLE_RATE)
        out.append(spec.numpy().astype(np.float16))
    return np.stack(out, axis=0)


# Build noise pool sized to ~NOISE_FRACTION of the eventual training set
n_noise = int(NOISE_FRACTION * total_train / (1 - NOISE_FRACTION))
NOISE_CACHE_PATH = CACHE_DIR / 'noise_pool.npy'
if NOISE_CACHE_PATH.exists():
    noise_pool = np.load(NOISE_CACHE_PATH)
    print(f'Loaded {noise_pool.shape[0]} cached noise samples.')
else:
    print(f'Generating {n_noise} noise samples ...')
    noise_pool = make_noise_pool(n_noise)
    np.save(NOISE_CACHE_PATH, noise_pool)
print(f'Noise pool: {noise_pool.shape}  ({noise_pool.dtype})')

In [ ]:
# ── Build class label space + flat (spec, label) index ───────────────────────
CLASS_NOISE = 0
class_to_idx = {('__noise__', '__noise__'): CLASS_NOISE}
for lang in LANGUAGES:
    for word in inventory[lang]['training']:
        if word in train_cache.get(lang, {}):
            class_to_idx[(lang, word)] = len(class_to_idx)
idx_to_class = {v: k for k, v in class_to_idx.items()}
N_CLASSES = len(class_to_idx)

print(f'Total embedding classes: {N_CLASSES} (incl. noise)')

lang_to_idx = {l: i for i, l in enumerate(LANGUAGES)}

train_index = []
for lang, by_word in train_cache.items():
    li = lang_to_idx[lang]
    for word, arr in by_word.items():
        cls = class_to_idx.get((lang, word))
        if cls is None:
            continue
        for j in range(arr.shape[0]):
            train_index.append((li, word, j, cls))

for j in range(noise_pool.shape[0]):
    train_index.append((-1, '__noise__', j, CLASS_NOISE))

random.Random(SEED).shuffle(train_index)
print(f'Total flat training examples (incl. noise): {len(train_index):,}')

split = int(0.9 * len(train_index))
train_records = train_index[:split]
val_records   = train_index[split:]
print(f'Train: {len(train_records):,}   Val: {len(val_records):,}')

# ── Balanced sampler (Mazumder et al. recipe) ─────────────────────────────────
# Each sample is assigned weight 1 / class_size so every class contributes
# equally to each batch regardless of how many clips it has.  This is the
# standard mitigation for the low-resource language imbalance: a Kinyarwanda
# word with 40 clips gets the same expected gradient contribution per epoch
# as an English word with 400 clips.
class_counts = Counter(cls for _, _, _, cls in train_records)
sample_weights = torch.tensor(
    [1.0 / class_counts[cls] for _, _, _, cls in train_records],
    dtype=torch.float64,
)
balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_records),
    replacement=True,
)

print('\nClass size range (train split):')
sizes = sorted(class_counts.values())
print(f'  min={sizes[0]}  median={sizes[len(sizes)//2]}  max={sizes[-1]}')
print(f'  Weight ratio max/min: {sizes[-1]/max(sizes[0],1):.1f}× '
      f'→ balanced sampler equalises this')


class EmbeddingDataset(Dataset):
    def __init__(self, records, augment: bool = False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        li, word, j, cls = self.records[idx]
        if li == -1:
            spec = noise_pool[j]
        else:
            spec = train_cache[LANGUAGES[li]][word][j]
        spec = torch.from_numpy(spec.astype(np.float32))
        if self.augment:
            spec = spec_augment(spec)
        return spec, int(cls), int(li)


train_ds = EmbeddingDataset(train_records, augment=True)
val_ds   = EmbeddingDataset(val_records,   augment=False)

# Training uses the balanced sampler (mutually exclusive with shuffle=True).
# Validation uses sequential order — no balancing needed for evaluation.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=balanced_sampler,
                          num_workers=0, pin_memory=(device.type == 'cuda'),
                          drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(device.type == 'cuda'))
print(f'Batches — train: {len(train_loader)}   val: {len(val_loader)}')

---
## Section 5 — DS-CNN backbone (256-dim multilingual embedding)

### Depthwise-separable convolution refresher

A standard Conv2d with kernel $k{\times}k$, $C_{in}$ input, $C_{out}$ output channels costs $O(k^2 C_{in} C_{out})$ per spatial location. A **depthwise separable block** factors this into:

- **Depthwise**: $k{\times}k$ filter applied independently per channel → $O(k^2 C_{in})$
- **Pointwise**: $1{\times}1$ conv to mix channels → $O(C_{in} C_{out})$

Total: $O(k^2 C_{in} + C_{in} C_{out})$ — roughly **8-9× cheaper** than standard conv for $k=3, C=64$. This is why DS-CNN became the standard TinyML KWS backbone (Zhang et al. 2017, "Hello Edge").

### Architecture

```
Input: (B, 1, 49, 40)

Conv2d(1→48, 3×3)        + BN + ReLU
DSBlock(48→64)
DSBlock(64→64)           MaxPool(2×2)
DSBlock(64→128, stride=2)
DSBlock(128→128)
DSBlock(128→192)
DSBlock(192→256)         GlobalAvgPool
                         ──▶  (B, 256)   ← THE EMBEDDING
                         Linear(256 → N_CLASSES)   ← training head (discarded at deploy)
```

The **penultimate 256-dim vector is the deployable artifact**. The final softmax exists only to give the embedding a training signal; we delete it at deployment and replace it with the tiny 5-shot heads from Section 8.

### Why these dimensions
- 48 → 256 channel ramp keeps the param count around 400-500K
- 256-dim embedding: Lei et al. 2023 swept this hyperparameter and found 256-512 is the sweet spot for medium backbones; 1024 actually hurt
- Two stride-2 reductions: spatial map goes 49×40 → 24×20 → 12×10 before GAP — enough resolution to localize phoneme structure

In [ ]:
class DSBlock(nn.Module):
    """Depthwise-separable conv block: DW(3x3)-BN-ReLU -> PW(1x1)-BN-ReLU."""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1,
                             groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.dw(x)))
        x = F.relu(self.bn2(self.pw(x)))
        return x


class MultilingualEmbedding(nn.Module):
    """
    DS-CNN backbone → 256-dim embedding → N-way softmax (training only).

    forward(x)        → logits, embedding
    embed(x)          → embedding only  (used post-freeze for Sections 7-8)
    """

    def __init__(self, n_classes: int, embed_dim: int = EMBEDDING_DIM):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 48, 3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            DSBlock(48, 64),
            DSBlock(64, 64),
            nn.MaxPool2d(2),
            DSBlock(64, 128, stride=2),
            DSBlock(128, 128),
            DSBlock(128, 192),
            DSBlock(192, 256),
            nn.AdaptiveAvgPool2d(1),
        )
        # Projection to embedding dim (cheap; lets us decouple GAP output from embed size)
        self.project = nn.Linear(256, embed_dim, bias=False)
        self.embed_bn = nn.BatchNorm1d(embed_dim)
        self.classifier = nn.Linear(embed_dim, n_classes)

    def embed(self, x):
        h = self.backbone(x).flatten(1)         # (B, 256)
        h = self.project(h)                     # (B, embed_dim)
        h = self.embed_bn(h)
        return h

    def forward(self, x):
        emb = self.embed(x)
        logits = self.classifier(emb)
        return logits, emb

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = MultilingualEmbedding(n_classes=N_CLASSES, embed_dim=EMBEDDING_DIM).to(device)
print(f'Embedding model params : {model.count_params():,}')
print(f'Backbone + project only: {sum(p.numel() for n,p in model.named_parameters() if not n.startswith("classifier")):,}')

# Shape sanity check
dummy = torch.randn(4, 1, 49, 40, device=device)
logits, emb = model(dummy)
print(f'Logits shape: {tuple(logits.shape)}  (expect [4, {N_CLASSES}])')
print(f'Embed  shape: {tuple(emb.shape)}     (expect [4, {EMBEDDING_DIM}])')

---
## Section 6 — Embedding training

**Loss**: cross-entropy with label smoothing (0.1) over the N classes. Label smoothing pushes the model away from over-confident outputs, which empirically yields better embeddings for downstream few-shot transfer (Müller et al., NeurIPS 2019 — "When does label smoothing help?").

**Optimizer**: AdamW + cosine LR with warm-up. Standard recipe; no bespoke tuning.

**Evaluation during training**: top-1 accuracy overall + **per-language top-1 accuracy** (Mazumder Table 1 reproduction). The per-language breakdown surfaces low-resource issues early — if Persian or Kinyarwanda accuracy is dramatically below the others, we can decide whether to drop them before sinking time into 5-shot eval.

**Checkpointing**: save the best-val-accuracy state to `CKPT_PATH`. Section 7-8 load this checkpoint.

In [ ]:
@torch.no_grad()
def evaluate_embedding(loader, model) -> dict:
    """Returns dict with 'overall' top-1 acc and 'per_lang' dict."""
    model.eval()
    correct_total = total_total = 0
    correct_per_lang = defaultdict(int)
    total_per_lang   = defaultdict(int)

    for specs, labels, lang_idx in loader:
        specs, labels = specs.to(device), labels.to(device)
        logits, _ = model(specs)
        pred = logits.argmax(1)
        right = (pred == labels)
        correct_total += right.sum().item()
        total_total   += labels.size(0)
        for li in lang_idx.unique().tolist():
            mask = lang_idx == li
            correct_per_lang[li] += right[mask.to(device)].sum().item()
            total_per_lang[li]   += mask.sum().item()

    per_lang = {}
    for li, n in total_per_lang.items():
        per_lang[li] = correct_per_lang[li] / max(n, 1)
    return {'overall': correct_total / max(total_total, 1), 'per_lang': per_lang}


def train_embedding(model, train_loader, val_loader, epochs=EPOCHS,
                    lr=LR, weight_decay=WEIGHT_DECAY,
                    label_smooth=LABEL_SMOOTH, ckpt_path=CKPT_PATH) -> dict:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=len(train_loader), pct_start=0.1,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    history   = {'loss': [], 'train_acc': [], 'val_acc': [], 'val_per_lang': []}
    best_val  = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = correct = total = 0
        t0 = time.time()
        for specs, labels, _ in train_loader:
            specs, labels = specs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            logits, _ = model(specs)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)

        train_loss = epoch_loss / len(train_loader)
        train_acc  = correct / total
        val_stats  = evaluate_embedding(val_loader, model)

        history['loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_stats['overall'])
        history['val_per_lang'].append(val_stats['per_lang'])

        if val_stats['overall'] > best_val:
            best_val = val_stats['overall']
            torch.save({
                'state_dict': model.state_dict(),
                'class_to_idx': {f'{k[0]}|{k[1]}': v for k, v in class_to_idx.items()},
                'n_classes': N_CLASSES,
                'embed_dim': EMBEDDING_DIM,
                'epoch': epoch,
                'val_acc': best_val,
            }, ckpt_path)

        dt = time.time() - t0
        print(f'Ep {epoch:3d}/{epochs} | loss {train_loss:.3f} | '
              f'train {train_acc:.3f} | val {val_stats["overall"]:.3f} | '
              f'best {best_val:.3f} | {dt:.1f}s')

    print(f'\nBest val accuracy: {best_val:.3f}  (checkpoint: {ckpt_path})')
    # Restore best state
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['state_dict'])
    return history


history = train_embedding(model, train_loader, val_loader)

In [ ]:
# ── Training curves + per-language final accuracy ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
eps = range(1, len(history['loss']) + 1)

axes[0].plot(eps, history['loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(eps, history['train_acc'], label='Train')
axes[1].plot(eps, history['val_acc'],   label='Val')
axes[1].set_title('Top-1 Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Final per-language accuracy (Mazumder Table 1 reproduction)
final_per_lang = history['val_per_lang'][-1]
ordered = [(LANGUAGES[li], acc) for li, acc in sorted(final_per_lang.items())
           if li >= 0]   # skip noise sentinel
langs, accs = zip(*ordered) if ordered else ([], [])
axes[2].barh(list(langs), list(accs), color=sns.color_palette('viridis', len(langs)))
axes[2].set_xlim(0, 1); axes[2].set_xlabel('Top-1 val accuracy')
axes[2].set_title('Per-language accuracy (final epoch)')
for i, a in enumerate(accs):
    axes[2].text(a + 0.01, i, f'{a:.2f}', va='center')

plt.tight_layout(); plt.savefig(CACHE_DIR / 'training_curves.png', dpi=150); plt.show()
print('\nFinal per-language accuracy:')
for lang_code, acc in ordered:
    print(f'  {lang_code:4s} {LANG_NAMES[lang_code]:12s}  {acc:.3f}')

---
## Section 7 — Embedding diagnostics (t-SNE by language)

We compute embeddings on a balanced sample of validation clips (~30 per language) and project to 2-D with t-SNE.

**What we expect**:
- If the embedding has learned language identity, clips cluster by language color
- If it has *also* learned acoustic-phonetic content (the deeper goal), similar-sounding words across languages should sit near each other regardless of color — that's what enables 5-shot generalization in Section 8

This plot replaces v1's "misrouting" experiment. The intuition (does the model's representation respect language structure?) is the same; the artifact is cleaner and standard.

In [ ]:
@torch.no_grad()
def collect_embeddings(records, model, max_per_lang=30):
    model.eval()
    pools = defaultdict(list)
    for li, word, j, cls in records:
        if li < 0:
            continue
        pools[li].append((word, j))

    feats, langs, words = [], [], []
    for li, items in pools.items():
        random.Random(SEED + li).shuffle(items)
        for word, j in items[:max_per_lang]:
            spec = torch.from_numpy(train_cache[LANGUAGES[li]][word][j].astype(np.float32))
            emb  = model.embed(spec.unsqueeze(0).to(device)).cpu().numpy().squeeze(0)
            feats.append(emb)
            langs.append(li)
            words.append(word)
    return np.stack(feats), np.array(langs), words


feats, langs, words = collect_embeddings(val_records, model, max_per_lang=30)
print(f'Embedding pool for t-SNE: {feats.shape}')

# 2-D projection
tsne = TSNE(n_components=2, perplexity=30, init='pca',
            learning_rate='auto', random_state=SEED)
xy = tsne.fit_transform(feats)

# Plot
plt.figure(figsize=(9, 7))
palette = sns.color_palette('tab10', len(LANGUAGES))
for li in range(len(LANGUAGES)):
    mask = langs == li
    if mask.any():
        plt.scatter(xy[mask, 0], xy[mask, 1], s=25, alpha=0.7,
                    color=palette[li], label=f'{LANGUAGES[li]} ({LANG_NAMES[LANGUAGES[li]]})')
plt.legend(loc='best', fontsize=9)
plt.title('t-SNE of validation embeddings, colored by language')
plt.xlabel('t-SNE dim 1'); plt.ylabel('t-SNE dim 2')
plt.tight_layout(); plt.savefig(CACHE_DIR / 'tsne_languages.png', dpi=150); plt.show()

---
## Section 8 — 5-shot head training & evaluation (headline experiment)

### Setup

For each `(language, held-out target word)` pair:

1. **Freeze** the embedding (no gradient flows back).
2. **Compute embeddings** for: 5 target samples + a balanced non-target pool drawn from MSWC clips of other words.
3. **Train a tiny 3-class softmax head** `Linear(256 → 3)` (target / unknown / background) for ~40 iterations on:
    - 5 target embeddings (positive class)
    - `N_UNKNOWN_PER_HEAD=128` non-target embeddings (unknown class)
    - `N_NOISE_PER_HEAD=25` noise embeddings (background class)
4. **Evaluate** on the remaining target samples (positive examples) and a fresh non-target pool (negative examples). Report F1 at threshold 0.8 (Mazumder convention).

### Why this experimental design

- **Out-of-embedding evaluation**: the held-out target keywords were not in the 450-class training set. So we're testing genuine generalization, not memorization.
- **5 samples** is the few-shot regime — what a real product flow looks like (user records 5 examples of "Hey, Toaster!" and the device starts listening for it).
- **Threshold = 0.8** lets us report F1 at a useful operating point; ROC-AUC is reported too for shape information.

### Expected result

Mazumder et al. report mean F1 ≈ 0.75 with their 11M-param EfficientNet embedding. Our DS-CNN is 20× smaller. **Anything in the 0.55-0.70 range is a strong outcome for our parameter budget**; matching their 0.75 would be a genuinely surprising result.

In [ ]:
# ── Helpers for 5-shot head training ──────────────────────────────────────────
TARGET_LBL, UNKNOWN_LBL, BG_LBL = 0, 1, 2

@torch.no_grad()
def embed_batch(specs: np.ndarray, model) -> np.ndarray:
    """specs: (N, 1, 49, 40) float16 -> embeddings (N, D) float32."""
    x = torch.from_numpy(specs.astype(np.float32)).to(device)
    return model.embed(x).cpu().numpy()


def build_unknown_pool(target_lang: str, target_word: str,
                       n_samples: int, model) -> np.ndarray:
    """
    Build a non-target pool by drawing from MSWC heldout words in OTHER languages
    + heldout words in the same language other than target_word.
    """
    pool_specs = []
    for lang in LANGUAGES:
        for word, arr in heldout_cache.get(lang, {}).items():
            if lang == target_lang and word == target_word:
                continue
            # take a few per word
            k = min(4, arr.shape[0])
            pool_specs.append(arr[:k])
    pool = np.concatenate(pool_specs, axis=0)
    # sample exactly n_samples
    rng = np.random.RandomState(SEED + hash(target_word) % 10_000)
    idx = rng.choice(pool.shape[0], size=min(n_samples, pool.shape[0]), replace=False)
    return embed_batch(pool[idx], model)


def train_fewshot_head(target_embs: np.ndarray, unknown_embs: np.ndarray,
                       bg_embs: np.ndarray, embed_dim: int = EMBEDDING_DIM,
                       epochs: int = HEAD_EPOCHS, lr: float = HEAD_LR) -> nn.Module:
    """
    Tiny 3-class linear head trained on top of frozen embeddings.
    Returns the trained head (on `device`).
    """
    X = np.concatenate([target_embs, unknown_embs, bg_embs], axis=0)
    y = np.concatenate([
        np.full(target_embs.shape[0],  TARGET_LBL,  dtype=np.int64),
        np.full(unknown_embs.shape[0], UNKNOWN_LBL, dtype=np.int64),
        np.full(bg_embs.shape[0],      BG_LBL,      dtype=np.int64),
    ])
    # Weight targets up to compensate for the imbalance (5 vs 128 vs 25)
    class_weight = torch.tensor([
        (X.shape[0] / 3.0) / max(target_embs.shape[0], 1),
        (X.shape[0] / 3.0) / max(unknown_embs.shape[0], 1),
        (X.shape[0] / 3.0) / max(bg_embs.shape[0], 1),
    ], dtype=torch.float32, device=device)

    head = nn.Linear(embed_dim, 3).to(device)
    opt  = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=class_weight)
    Xt = torch.from_numpy(X).to(device)
    yt = torch.from_numpy(y).to(device)
    for _ in range(epochs):
        opt.zero_grad()
        loss = crit(head(Xt), yt)
        loss.backward(); opt.step()
    return head


def eval_fewshot_head(head: nn.Module, target_test_embs: np.ndarray,
                      negative_test_embs: np.ndarray, threshold: float = F1_THRESHOLD) -> dict:
    """
    Returns dict with f1, precision, recall, roc_auc, tpr/fpr at threshold.
    Positive = target class. Decision = (P[target] > threshold).
    """
    head.eval()
    with torch.no_grad():
        pos_logits = head(torch.from_numpy(target_test_embs).to(device))
        neg_logits = head(torch.from_numpy(negative_test_embs).to(device))
    pos_prob = F.softmax(pos_logits, dim=1)[:, TARGET_LBL].cpu().numpy()
    neg_prob = F.softmax(neg_logits, dim=1)[:, TARGET_LBL].cpu().numpy()

    y_true   = np.concatenate([np.ones_like(pos_prob), np.zeros_like(neg_prob)])
    y_score  = np.concatenate([pos_prob, neg_prob])
    y_pred   = (y_score >= threshold).astype(int)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc     = auc(fpr, tpr)
    except Exception:
        fpr, tpr, roc_auc = np.array([0., 1.]), np.array([0., 1.]), float('nan')
    return {'f1': f1, 'precision': prec, 'recall': rec, 'roc_auc': roc_auc,
            'tpr_at_thr': rec, 'fpr_at_thr': fp / max(fp + tn, 1),
            'roc_curve': (fpr, tpr)}


print('5-shot helpers defined.')

In [ ]:
# ── Sweep across (lang, heldout-word) pairs ───────────────────────────────────
# For each, precompute target embeddings once (cheap), then run train+eval.

fewshot_results = []   # list of dicts
n_pairs = 0
for lang in LANGUAGES:
    for word in inventory[lang]['heldout']:
        if word not in heldout_cache.get(lang, {}):
            continue
        n_pairs += 1
print(f'Sweeping {n_pairs} (lang, heldout-word) pairs...')

# Build a shared embedded background pool (noise) once
bg_embs_full = embed_batch(noise_pool[:max(N_NOISE_PER_HEAD * 4, 100)], model)

t_start = time.time()
for lang in LANGUAGES:
    for word in inventory[lang]['heldout']:
        if word not in heldout_cache.get(lang, {}):
            continue
        arr = heldout_cache[lang][word]
        if arr.shape[0] < N_SHOT + 30:
            continue   # need enough left over for eval
        # Random 5-shot picks
        rng = np.random.RandomState(SEED + hash(f'{lang}|{word}') % 10_000)
        perm = rng.permutation(arr.shape[0])
        shot_idx = perm[:N_SHOT]
        test_idx = perm[N_SHOT:N_SHOT + 100]

        target_train_embs = embed_batch(arr[shot_idx], model)
        target_test_embs  = embed_batch(arr[test_idx], model)
        unknown_embs      = build_unknown_pool(lang, word, N_UNKNOWN_PER_HEAD, model)
        bg_idx = rng.choice(bg_embs_full.shape[0],
                            size=min(N_NOISE_PER_HEAD, bg_embs_full.shape[0]),
                            replace=False)
        bg_embs = bg_embs_full[bg_idx]

        head = train_fewshot_head(target_train_embs, unknown_embs, bg_embs)

        # Negative test pool = fresh draw from non-target heldout in all langs
        neg_pool_embs = build_unknown_pool(lang, word, 200, model)
        metrics = eval_fewshot_head(head, target_test_embs, neg_pool_embs)
        metrics['lang']  = lang
        metrics['word']  = word
        metrics['n_test_pos'] = int(target_test_embs.shape[0])
        fewshot_results.append(metrics)

print(f'\n5-shot sweep done in {time.time()-t_start:.1f}s. '
      f'{len(fewshot_results)} keywords evaluated.')

# Summary by language
print(f'\n{"lang":6s} {"#kw":>4s} {"mean F1":>9s} {"median F1":>11s} {"mean AUC":>10s}')
per_lang_summary = []
for lang in LANGUAGES:
    items = [r for r in fewshot_results if r['lang'] == lang]
    if not items:
        continue
    f1s  = [r['f1']      for r in items]
    aucs = [r['roc_auc'] for r in items if not math.isnan(r['roc_auc'])]
    per_lang_summary.append((lang, len(items), np.mean(f1s), np.median(f1s), np.mean(aucs)))
    print(f'{lang:6s} {len(items):4d} {np.mean(f1s):9.3f} {np.median(f1s):11.3f} {np.mean(aucs):10.3f}')

all_f1   = [r['f1']      for r in fewshot_results]
all_aucs = [r['roc_auc'] for r in fewshot_results if not math.isnan(r['roc_auc'])]
print(f'\nOverall mean F1 @ {F1_THRESHOLD:.1f}: {np.mean(all_f1):.3f}  '
      f'(Mazumder 11M EfficientNet: 0.75)')
print(f'Overall mean ROC-AUC          : {np.mean(all_aucs):.3f}')

In [ ]:
# ── Visualize 5-shot results (Mazumder Fig. 2 style) ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (a) F1 distribution per language
df_rows = []
for r in fewshot_results:
    df_rows.append({'lang': r['lang'], 'F1': r['f1'], 'AUC': r['roc_auc']})
langs_present = sorted({r['lang'] for r in fewshot_results}, key=LANGUAGES.index)
palette = sns.color_palette('tab10', len(LANGUAGES))
lang_color = {LANGUAGES[i]: palette[i] for i in range(len(LANGUAGES))}

# seaborn >=0.13 requires DataFrame or Mapping (not list of dicts)
df_rows = {
    'lang': [row['lang'] for row in df_rows],
    'F1':   [row['F1'] for row in df_rows],
    'AUC':  [row['AUC'] for row in df_rows],
}

sns.boxplot(
    x='lang', y='F1', data=df_rows, order=langs_present,
    palette=[lang_color[l] for l in langs_present], ax=axes[0],
)
sns.stripplot(
    x='lang', y='F1', data=df_rows, order=langs_present,
    color='black', size=3, alpha=0.5, ax=axes[0],
)
axes[0].axhline(0.75, color='red', linestyle='--', alpha=0.6,
                label='Mazumder (11M EffNet)')
axes[0].set_ylim(0, 1.0); axes[0].set_title(f'5-shot F1 @ threshold {F1_THRESHOLD} per language')
axes[0].legend()

# (b) ROC curves overlay per language (mean over its keywords)
for lang in langs_present:
    items = [r for r in fewshot_results if r['lang'] == lang]
    # Common FPR grid
    grid = np.linspace(0, 1, 200)
    tprs = []
    for r in items:
        fpr, tpr = r['roc_curve']
        tprs.append(np.interp(grid, fpr, tpr))
    if not tprs:
        continue
    mean_tpr = np.mean(tprs, axis=0)
    axes[1].plot(grid, mean_tpr, color=lang_color[lang],
                 label=f'{lang} ({LANG_NAMES[lang]}) AUC={np.mean([r["roc_auc"] for r in items]):.2f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].set_title('Mean ROC per language (5-shot)')
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout(); plt.savefig(CACHE_DIR / 'fewshot_results.png', dpi=150); plt.show()

# Save results table for the report
results_path = CACHE_DIR / 'fewshot_results.json'
results_path.write_text(json.dumps(
    [{'lang': r['lang'], 'word': r['word'], 'f1': r['f1'],
      'roc_auc': r['roc_auc'], 'precision': r['precision'],
      'recall': r['recall']} for r in fewshot_results],
    indent=2, ensure_ascii=False,
), encoding='utf-8')
print(f'Results table saved: {results_path}')

---
## Section 8b — Optional: TTS-enrolled heads (Lin et al. reproduction)

**Skip this section if Colab time is tight — it's a supplement, not part of the headline.**

The Section 8 5-shot eval uses *real* MSWC clips as the 5 target samples. But in a real deployment, a user might want to teach the device a brand-new wake word that doesn't exist in any corpus (a product name, an invented phrase). The natural answer is **TTS-generated samples**.

Lin et al. (ICASSP 2020) showed that a **head model on top of a real-audio embedding can be trained entirely on synthetic speech** and reach 92.6% on a real-audio test set — within 2% of the same head trained on real speech. We reproduce that finding here with our small-backbone embedding: for each test keyword, we generate 5 TTS samples (using the existing `generate_tts.py` infrastructure) and train the head on those instead of MSWC samples. The held-out **test set remains real MSWC clips**.

Result we expect: 5-shot F1 from TTS-enrolled heads should be **within ~5% of real-enrolled F1** for languages with diverse TTS voices, validating the "5 TTS samples can replace 5 real samples" deployment claim.

The code below is wrapped in a `try/except` — if `edge_tts` isn't available, the section is skipped without affecting Section 9.

In [ ]:
# Optional: TTS-enrolled heads. Requires `edge-tts` (pip install edge-tts).
# We pick a small handful of representative heldout words across languages
# and compare TTS-enrolled vs real-enrolled 5-shot F1 head-to-head.

TTS_VOICES = {
    'en': ['en-US-AriaNeural', 'en-US-GuyNeural', 'en-GB-RyanNeural',
           'en-GB-SoniaNeural', 'en-AU-WilliamNeural'],
    'de': ['de-DE-KatjaNeural', 'de-DE-ConradNeural', 'de-AT-IngridNeural',
           'de-CH-LeniNeural', 'de-DE-AmalaNeural'],
    'fr': ['fr-FR-DeniseNeural', 'fr-FR-HenriNeural', 'fr-CA-SylvieNeural',
           'fr-CA-AntoineNeural', 'fr-CH-ArianeNeural'],
    'es': ['es-ES-ElviraNeural', 'es-ES-AlvaroNeural', 'es-MX-DaliaNeural',
           'es-MX-JorgeNeural', 'es-AR-ElenaNeural'],
    'it': ['it-IT-ElsaNeural', 'it-IT-IsabellaNeural', 'it-IT-DiegoNeural'],
    'nl': ['nl-NL-FennaNeural', 'nl-NL-MaartenNeural', 'nl-BE-DenaNeural'],
    'ca': ['ca-ES-JoanaNeural', 'ca-ES-EnricNeural'],
    'fa': ['fa-IR-DilaraNeural', 'fa-IR-FaridNeural'],
    # rw (Kinyarwanda) is not supported by edge-tts; skipped automatically.
}

def tts_enroll_one(lang, word, model, n_shot=N_SHOT):
    """Synthesize n_shot TTS samples for (lang, word) and return their embeddings."""
    import asyncio, edge_tts, tempfile

    voices = TTS_VOICES.get(lang, [])
    if not voices:
        return None  # unsupported language
    voices = voices[:n_shot]
    embs = []
    for voice in voices:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            outpath = f.name
        async def _gen():
            comm = edge_tts.Communicate(text=word, voice=voice)
            await comm.save(outpath)
        asyncio.run(_gen())
        wav, sr = torchaudio.load(outpath)
        spec = waveform_to_logmel(wav.squeeze(0), sr)
        embs.append(model.embed(spec.unsqueeze(0).to(device)).cpu().numpy().squeeze(0))
        os.remove(outpath)
    return np.stack(embs).astype(np.float32)


tts_results = []
try:
    import edge_tts   # noqa: F401
    print('Running TTS-enrolled head experiment ...')

    # Pick up to 5 keywords per supported lang for the comparison
    comparison_pairs = []
    for lang in LANGUAGES:
        if lang not in TTS_VOICES:
            continue
        kws = [w for w in inventory[lang]['heldout']
               if w in heldout_cache.get(lang, {})]
        comparison_pairs.extend((lang, w) for w in kws[:5])
    print(f'  {len(comparison_pairs)} comparison keywords across {len(TTS_VOICES)} TTS-supported languages.')

    for lang, word in comparison_pairs:
        tts_embs = tts_enroll_one(lang, word, model)
        if tts_embs is None:
            continue
        arr = heldout_cache[lang][word]
        rng = np.random.RandomState(SEED + hash(f'{lang}|{word}|tts') % 10_000)
        test_idx = rng.permutation(arr.shape[0])[:100]
        target_test_embs = embed_batch(arr[test_idx], model)
        unknown_embs     = build_unknown_pool(lang, word, N_UNKNOWN_PER_HEAD, model)
        bg_idx = rng.choice(bg_embs_full.shape[0],
                            size=min(N_NOISE_PER_HEAD, bg_embs_full.shape[0]),
                            replace=False)
        bg_embs = bg_embs_full[bg_idx]
        head = train_fewshot_head(tts_embs, unknown_embs, bg_embs)
        neg_pool_embs = build_unknown_pool(lang, word, 200, model)
        m = eval_fewshot_head(head, target_test_embs, neg_pool_embs)
        m.update({'lang': lang, 'word': word, 'enrollment': 'TTS'})
        tts_results.append(m)
        print(f'  TTS  {lang}/{word:20s} F1={m["f1"]:.3f}  AUC={m["roc_auc"]:.3f}')

    # Compare side-by-side with real-enrolled results on the same keywords
    print(f'\n{"lang/word":30s} {"F1 real":>8s} {"F1 TTS":>8s} {"Δ":>7s}')
    comp_real = {(r['lang'], r['word']): r['f1'] for r in fewshot_results}
    deltas = []
    for r in tts_results:
        key = (r['lang'], r['word'])
        real_f1 = comp_real.get(key)
        if real_f1 is None:
            continue
        delta = r['f1'] - real_f1
        deltas.append(delta)
        print(f'{r["lang"]}/{r["word"]:25s} {real_f1:8.3f} {r["f1"]:8.3f} {delta:+7.3f}')
    if deltas:
        print(f'\nMean F1 delta (TTS - real): {np.mean(deltas):+.3f}')

except ImportError:
    print('edge-tts not installed. Section 8b skipped.')
    print('Run `pip install edge-tts` to enable.')
except Exception as exc:
    print(f'Section 8b failed (non-fatal): {exc}')

---
## Section 9 — Save & summary

We save:
- The trained embedding (float32 PyTorch state dict) — input to `multilingual_kws_export.ipynb`
- The inventory + class-to-index mapping (so the export notebook reconstructs the architecture)
- The 5-shot results JSON (for the report)
- Plots: training curves, t-SNE, 5-shot ROC

A separate `multilingual_kws_export.ipynb` notebook will pick up the checkpoint and produce the int8 ONNX deployment artifact + ESP32 budget table.

In [ ]:
# ── Final checkpoint bundle ───────────────────────────────────────────────────
bundle_path = CACHE_DIR / 'embedding_final.pt'
torch.save({
    'state_dict':   model.state_dict(),
    'n_classes':    N_CLASSES,
    'embed_dim':    EMBEDDING_DIM,
    'class_to_idx': {f'{k[0]}|{k[1]}': v for k, v in class_to_idx.items()},
    'inventory':    inventory,
    'history':      history,
    'fewshot_summary': {
        'overall_mean_f1':   float(np.mean(all_f1)),
        'overall_mean_auc':  float(np.mean(all_aucs)),
        'per_lang':          [
            {'lang': l, 'n_kw': n, 'mean_f1': mf, 'median_f1': medf, 'mean_auc': ma}
            for l, n, mf, medf, ma in per_lang_summary
        ],
    },
    'config': {
        'SAMPLE_RATE': SAMPLE_RATE, 'N_MELS': N_MELS,
        'N_FFT': N_FFT, 'WIN_LENGTH': WIN_LENGTH, 'HOP_LENGTH': HOP_LENGTH,
        'TOP_K_PER_LANG': TOP_K_PER_LANG,
        'SAMPLES_PER_CLASS': SAMPLES_PER_CLASS,
        'N_SHOT': N_SHOT, 'F1_THRESHOLD': F1_THRESHOLD,
    },
}, bundle_path)
print(f'Checkpoint bundle saved: {bundle_path}  ({bundle_path.stat().st_size/1e6:.1f} MB float32)')

# Summary of all artefacts
artefacts = [
    CACHE_DIR / 'keyword_inventory.json',
    CACHE_DIR / 'embedding_best.pt',
    CACHE_DIR / 'embedding_final.pt',
    CACHE_DIR / 'training_curves.png',
    CACHE_DIR / 'tsne_languages.png',
    CACHE_DIR / 'fewshot_results.png',
    CACHE_DIR / 'fewshot_results.json',
]
print('\nArtefacts produced:')
for p in artefacts:
    if p.exists():
        print(f'  {str(p):60s} {p.stat().st_size/1e3:8.1f} KB')
    else:
        print(f'  {str(p):60s}  (not generated)')

# Quick text summary
print('\n' + '=' * 70)
print('SUMMARY')
print('=' * 70)
print(f'Backbone params         : {model.count_params():,}')
print(f'Embedding dim           : {EMBEDDING_DIM}')
print(f'Languages               : {len(LANGUAGES)}')
print(f'Training classes        : {N_CLASSES} (incl. noise)')
print(f'Best val top-1 accuracy : {max(history["val_acc"]):.3f}')
print(f'5-shot mean F1 @ {F1_THRESHOLD}    : {np.mean(all_f1):.3f}  '
      f'(Mazumder 11M EffNet baseline: 0.75)')
print(f'5-shot mean ROC-AUC     : {np.mean(all_aucs):.3f}')
print('=' * 70)
print('\nNext step: run multilingual_kws_export.ipynb to produce the int8 ONNX')
print('deployment artifact and the ESP32 budget table.')